In [ ]:
%pip install --break-system-packages psycopg2-binary sqlalchemy

import pandas as pd

from sqlalchemy import create_engine
from urllib.parse import quote_plus


In [ ]:
# CSV lives in ../data/ relative to this notebook's folder (repo: data/customer_shopping_behavior.csv)
df = pd.read_csv('../data/customer_shopping_behavior.csv')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include = 'all')

In [ ]:
df.isnull().sum()

In [ ]:
df['Review Rating'] = df.groupby('Category')['Review Rating'].transform(lambda x: x.fillna(x.median()))

In [ ]:
df.isnull().sum()

In [ ]:
df.columns = df.columns.str.lower()
df.columns = df.columns.str.replace(' ', '_')
df = df.rename(columns = {'purchase_amount_(usd)':'purchase_amount'})

In [ ]:
df.columns

In [ ]:
new_labels = ['Young Adult','Adult', 'Middle-aged', 'Senior']
df['age_group'] = pd.qcut(df['age'], q = 4, labels = new_labels)

In [ ]:
df[['age', 'age_group']].head(10)

In [ ]:
frequency_mapping = {
    'Fortnightly' : 14,
    'Weekly' : 7,
    'Monthly' : 30,
    'Quarterly' : 90,
    'Bi-Weekly' : 14,
    'Annually' : 365,
    'Every 3 Months' : 90
}
df['purchase_frequency_days'] = df['frequency_of_purchases'].map(frequency_mapping)

In [ ]:
df[['purchase_frequency_days', 'frequency_of_purchases']].head(10)

In [ ]:
df[['discount_applied', 'promo_code_used']].head(10)

In [ ]:
(df['discount_applied'] == df['promo_code_used']).all()

In [ ]:
df = df.drop('promo_code_used', axis = 1)

In [ ]:
df.columns

In [ ]:
import os

# Never hardcode credentials in a shared notebook.
# Set once in terminal:  setx PG_PASSWORD "your-password"   (then restart Jupyter)
username = "postgres"
password = os.environ["PG_PASSWORD"]          # reads from environment variable
host = 'localhost'
port = "5432"
database = "customer_behavior"

engine = create_engine(f"postgresql+psycopg2://{username}:{quote_plus(password)}@{host}:{port}/{database}")

table_name = "customer"
df.to_sql(table_name, engine, if_exists = "replace", index = False)

print(f"Data successfully loaded into table '{table_name}' in database '{database}' .")